Generate YouTube-style chapter timestamps for podcast episodes. WhisperX detects where speech starts and stops, then an LLM summarizes each segment into a chapter title.

The end result looks like this:

```
0:00 - Experiencing self vs remembering self
0:10 - Whether memories are the primary source of happiness
0:36 - Controlling how we remember experiences
```

## Problem

You have podcast episodes and want to generate chapter markers — the kind you see in YouTube descriptions or podcast apps. Each chapter needs a timestamp and a short description of what's being discussed.

## Solution

**What's in this recipe:**

1. **WhisperX** transcribes the audio and detects speech boundaries via VAD (Voice Activity Detection)
2. A **UDF** extracts the segments with their start times and transcribed text
3. **GPT-4o-mini** summarizes each segment's text into a short chapter title
4. The result is formatted as YouTube-style timestamps

### Setup

In [ ]:
%pip install -qU pixeltable whisperx openai

In [1]:
import getpass
import os

if 'OPENAI_API_KEY' not in os.environ:
    os.environ['OPENAI_API_KEY'] = getpass.getpass('OpenAI API Key: ')

In [2]:
import pixeltable as pxt
import pixeltable.functions as pxtf

### Load a podcast episode

In [3]:
pxt.drop_dir('podcast_demo', force=True, if_not_exists='ignore')
pxt.create_dir('podcast_demo')

Connected to Pixeltable database at: postgresql+psycopg://postgres:@/pixeltable?host=/Users/alison-pxt/.pixeltable/pgdata
Created directory 'podcast_demo'.


/Users/alison-pxt/miniconda3/envs/pxt313-whisperx/lib/python3.13/site-packages/pixeltable/env.py:387: UserWarning: Progress reporting is disabled because ipywidgets is not installed. To fix this, run: `pip install ipywidgets`
  warnings.warn(


In [4]:
episodes = pxt.create_table(
    'podcast_demo/episodes',
    {'title': pxt.String, 'audio': pxt.Audio}
)

Created table 'episodes'.


In [5]:
episodes.insert([{
    'title': 'Lex Fridman Podcast Excerpt',
    'audio': 'https://raw.githubusercontent.com/pixeltable/pixeltable/main/docs/resources/audio-transcription-demo/Lex-Fridman-Podcast-430-Excerpt-0.mp4'
}])

Inserted 1 row with 0 errors in 0.41 s (2.43 rows/s)


1 row inserted.

### Step 1: Transcribe with WhisperX

WhisperX does two things at once: it runs Voice Activity Detection (VAD) to find where speech occurs, then transcribes each speech region. The output is a list of segments — each with a start time, end time, and the text that was spoken.

These segments are the raw material for our chapter markers. Each segment boundary corresponds to a natural pause or transition in the conversation.

In [ ]:
episodes.add_computed_column(
    transcription=pxtf.whisperx.transcribe(episodes.audio, model='tiny.en')
)

In [7]:
episodes.select(
    segment_starts=episodes.transcription.segments['*'].start,
    segment_ends=episodes.transcription.segments['*'].end,
    segment_text=episodes.transcription.segments['*'].text
).collect()

segment_starts,segment_ends,segment_text
"[0.908, 10.73, 36.312]","[10.443, 36.194, 58.891]","["" of experiencing self versus remembering self. I was hoping you can give a simple answer of how we should live life."", "" Based on the fact that our memories could be a source of happiness or could be the primary source of happiness that an event when experienced bea ...... it's remembered over and over and over and over and maybe there is some wisdom and the fact that we can control to some degree how we remember it."", "" how we evolve our memory of it, such that it can maximize the long-term happiness of that repeated experience. Okay, well, first, I'll say, I wis ...... an I be your opening actor? Oh, my God. No, I've got to hope it for you, dude. Otherwise, it's like, you know, everybody leaves after you're done.""]"


The transcription output has a `segments` list inside it. We can pull out the start times, end times, and text for each segment using Pixeltable's JSON path expressions. Each row in the output below is a parallel array — the first start time goes with the first end time and first text, and so on.

Here's a more readable view of the same data — each line shows the timestamp range and a preview of what was said in that segment. Notice how the segments correspond to natural speech regions, not fixed time intervals:

In [8]:
for seg in episodes.collect()['transcription'][0]['segments']:
    mins, secs = divmod(int(seg['start']), 60)
    print(f"  {mins}:{secs:02d} - {seg['end']:.1f}s  {seg['text'].strip()[:90]}")

  0:00 - 10.4s  of experiencing self versus remembering self. I was hoping you can give a simple answer of
  0:10 - 36.2s  Based on the fact that our memories could be a source of happiness or could be the primary
  0:36 - 58.9s  how we evolve our memory of it, such that it can maximize the long-term happiness of that 


### Step 2: Generate chapter titles

The segments have timestamps and full transcribed text, but a chapter marker needs a short title — not the entire transcript. We'll define three UDFs and wire them together as computed columns:

- `extract_chapters` — Pull out each segment's start time and text
- `build_prompt` — Format the segment texts for the LLM  
- `generate_timestamps` — Combine start times with LLM-generated titles (used in Step 3)

First, we define and add the `chapters` column with `extract_chapters`, then define `generate_timestamps` (for Step 3):

In [9]:
@pxt.udf
def extract_chapters(transcription: dict) -> list[dict]:
    """Extract start time and text for each speech segment."""
    return [
        {
            'start': seg['start'],
            'text': seg.get('text', '').strip()
        }
        for seg in transcription.get('segments', [])
    ]

episodes.add_computed_column(chapters=extract_chapters(episodes.transcription), if_exists='replace')

Added 1 column value with 0 errors in 0.01 s (83.57 rows/s)


1 row updated.

In [10]:
@pxt.udf
def generate_timestamps(chapters: list[dict], titles: str) -> str:
    """Combine start times with LLM-generated chapter titles into YouTube-style timestamps."""
    title_list = [t.strip() for t in titles.strip().split('\n') if t.strip()]
    lines = []
    for i, ch in enumerate(chapters):
        mins, secs = divmod(int(ch['start']), 60)
        title = title_list[i] if i < len(title_list) else 'Untitled'
        lines.append(f"{mins}:{secs:02d} - {title}")
    return '\n'.join(lines)

Now wire them together as computed columns. The prompt is built from the chapter text, sent to GPT-4o-mini, and the response is extracted:

In [11]:
@pxt.udf
def build_prompt(chapters: list[dict]) -> str:
    """Build a prompt asking the LLM to title each segment."""
    segment_texts = '\n---\n'.join(
        f"Segment {i+1}: {ch['text']}" for i, ch in enumerate(chapters)
    )
    return (
        f"Below are {len(chapters)} consecutive segments from a podcast transcript. "
        "Write a short chapter title (5-10 words max) for each segment. "
        "Return one title per line, nothing else.\n\n"
        f"{segment_texts}"
    )

episodes.add_computed_column(prompt=build_prompt(episodes.chapters), if_exists='replace')

Added 1 column value with 0 errors in 0.02 s (41.14 rows/s)


1 row updated.

In [12]:
# Send to GPT-4o-mini for chapter titles
episodes.add_computed_column(
    title_response=pxtf.openai.chat_completions(
        messages=[{'role': 'user', 'content': episodes.prompt}],
        model='gpt-4o-mini',
    )
)

episodes.add_computed_column(
    titles=episodes.title_response.choices[0].message.content
)

Added 1 column value with 0 errors in 2.19 s (0.46 rows/s)
Added 1 column value with 0 errors in 0.01 s (102.92 rows/s)


1 row updated.

### Step 3: YouTube-style timestamps

Combine the segment start times with the LLM-generated chapter titles:

In [13]:
episodes.add_computed_column(
    timestamps=generate_timestamps(episodes.chapters, episodes.titles)
)

Added 1 column value with 0 errors in 0.04 s (28.09 rows/s)


1 row updated.

In [14]:
# The final result
print(episodes.collect()['timestamps'][0])

0:00 - Experiencing Self vs. Remembering Self
0:10 - The Power of Memory in Happiness
0:36 - Evolving Memories for Lasting Joy


## Explanation

**Pipeline:**

```
Audio → WhisperX (VAD + transcription) → segments → GPT-4o-mini → chapter titles → timestamps
```

Every step is a computed column. Insert a new episode and the entire pipeline runs automatically — from raw audio to YouTube-style timestamps.

**How WhisperX finds the chapter boundaries:**

WhisperX uses PyAnnote VAD to detect where speech occurs in the audio. Pauses, silence, and transitions between speakers create natural segment boundaries. These boundaries become the chapter start times.

**Trade-offs:**

- This approach requires running a full transcription model just to find where the pauses are
- For longer episodes, WhisperX's `chunk_size` parameter controls how the audio is batched internally
- The chapter titles depend on LLM quality — `gpt-4o-mini` is fast and cheap, use `gpt-4o` for higher quality

## See also

- [Transcribe audio](https://docs.pixeltable.com/howto/cookbooks/audio/audio-transcribe) — Fixed-interval splitting with local Whisper
- [Summarize podcasts](https://docs.pixeltable.com/howto/cookbooks/audio/audio-summarize-podcast) — Transcription + LLM summarization
- [Video scene detection](https://docs.pixeltable.com/howto/cookbooks/video/video-scene-detection) — Content-aware detection for video